# 02 — Cloud Cluster Simulator

Custom OpenAI Gymnasium environment for the RL cloud resource manager.
Generates realistic workloads from Google Cluster Trace 2011 patterns
(loaded from `trace_params.json`), models CPU **and** memory, tracks SLA
deadlines, and produces a 32-dim interval-robust state vector.

Run top-to-bottom.

## 1. Setup and constants

In [1]:
import json
import numpy as np
from env import CloudClusterEnv,STEPS_PER_WEEK

Setup complete. Steps per week: 672
CloudClusterEnv defined.


## 2. The environment

The full `CloudClusterEnv` in one place — CPU and memory, job generation,
VM packing (a job needs both CPU and memory room), SLA deadlines, reward,
32-dim state, decision log, and hint slots.

In [2]:
with open('trace_params.json') as f:
    stats = json.load(f)['stats']
print(type(stats))

<class 'dict'>


## 3. Sanity check — random agent

Runs one full week with random actions to confirm the environment works
end-to-end. A random agent should rack up many SLA breaches (it scales
badly) — that's expected and correct. The trained PPO agent will bring
this down dramatically.

In [3]:
env = CloudClusterEnv(stats)
obs, _ = env.reset()
print("Initial state shape:", obs.shape, "(should be 32)")
print("Running one week with random actions...\n")

total_reward = total_cost = total_breaches = 0
vm_counts = []
for t in range(STEPS_PER_WEEK):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    total_cost += info['cost']
    total_breaches += info['breaches']
    vm_counts.append(info['active_vms'])
    if done:
        break

print(f"Steps run:          {t+1}")
print(f"Total reward:       {total_reward:.1f}")
print(f"Total cost:         {total_cost:.1f}")
print(f"Total SLA breaches: {total_breaches}")
print(f"Avg VMs used:       {np.mean(vm_counts):.1f}")
print(f"Min/Max VMs:        {min(vm_counts)} / {max(vm_counts)}")
print(f"State is 32-dim:    {obs.shape == (32,)}")

Initial state shape: (32,) (should be 32)
Running one week with random actions...

Steps run:          672
Total reward:       -146.3
Total cost:         368.1
Total SLA breaches: 11690
Avg VMs used:       11.0
Min/Max VMs:        2 / 20
State is 32-dim:    True
